# Linear Regression

**难度:** Medium

用三种方式实现线性回归：闭式解、梯度下降和 nn.Linear。

本题涵盖拟合线性模型的基本方法，从正规方程到基于 autograd 的训练。

**签名:** `LinearRegression()`（类）

**方法:**
- `closed_form(X, y) -> (w, b)` — 通过正规方程求解
- `gradient_descent(X, y, lr, steps) -> (w, b)` — 手动梯度更新
- `nn_linear(X, y, lr, steps) -> (w, b)` — PyTorch autograd 训练循环

**约束:**
- 三种方法应收敛到相近的权重
- 闭式解不应使用 autograd
- X 为 (N, D)，y 为 (N,)，返回 w (D,) 和 b（标量）

**提示:**
closed_form：X 增加全 1 列 → `lstsq(X_aug, y)` → 拆分 `w, b`
gradient_descent：`grad_w = (2/N) * X.T @ (X@w+b - y)`，`w -= lr*grad_w`
nn_linear：`nn.Linear(D,1)` + `MSELoss` + `optimizer.step()` 循环

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ INTERVIEW

# ---- 手写线性回归三种实现 ----
# 面试要点：三种方法的数学推导 + PyTorch autograd 训练循环

class LinearRegression:
    # ---- 方法 1: 闭式解 ----
    # 面试考点：正规方程 θ = (X^T X)^{-1} X^T y
    # 优点：一步到位，不需要调参
    # 缺点：O(D^3) 复杂度，D 大时不可行；X^T X 可能不可逆
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        N, D = X.shape
        # 将偏置 b 并入权重：X_aug = [X, 1]
        X_aug = torch.cat([X, torch.ones(N, 1)], dim=1)
        # lstsq 比直接求逆更稳定
        theta = torch.linalg.lstsq(X_aug, y).solution
        w = theta[:D]
        b = theta[D]
        return w.detach(), b.detach()

    # ---- 方法 2: 梯度下降 ----
    # 面试考点：MSE 的梯度推导
    # L = (1/N) Σ (Xw+b-y)²
    # ∂L/∂w = (2/N) X^T (Xw+b-y)
    # ∂L/∂b = (2/N) Σ (Xw+b-y)
    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        N, D = X.shape
        w = torch.zeros(D)
        b = torch.tensor(0.0)

        for _ in range(steps):
            pred = X @ w + b
            error = pred - y
            grad_w = (2.0 / N) * (X.T @ error)
            grad_b = (2.0 / N) * error.sum()
            w = w - lr * grad_w
            b = b - lr * grad_b
        return w, b

    # ---- 方法 3: nn.Linear + autograd ----
    # 面试考点：标准的 PyTorch 训练循环
    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        N, D = X.shape
        layer = nn.Linear(D, 1)
        optimizer = torch.optim.SGD(layer.parameters(), lr=lr)
        loss_fn = nn.MSELoss()

        for _ in range(steps):
            optimizer.zero_grad()       # 1. 清零梯度
            pred = layer(X).squeeze(-1) # 2. 前向传播
            loss = loss_fn(pred, y)     # 3. 计算损失
            loss.backward()             # 4. 反向传播
            optimizer.step()            # 5. 更新参数

        w = layer.weight.data.squeeze(0)
        b = layer.bias.data.squeeze(0)
        return w, b

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("linear_regression")

## 📝 核心思路总结

1. **闭式解（正规方程）**：一步到位，但 O(D³) 复杂度，大维度不可行
2. **梯度下降**：手动推导梯度，理解 MSE 损失的导数 = (2/N) * X^T * error
3. **nn.Linear + autograd**：PyTorch 自动求梯度，最实用但面试要能手写